# Acuifero + Vigia - Live Hardware Replay Demo

Hardware replay demo for the Kaggle Gemma 4 Good Hackathon. Judges can verify
the alert orchestration without a Raspberry Pi, backend, frontend, GPU, or
internet downloads.

First-screen honesty checklist:

- This is a hardware replay demo, **not** a live Raspberry Pi stream.
- `RUN_GEMMA = False` by default; the notebook does **not** run LiteRT-LM or
  live Gemma inference unless you explicitly enable it.
- The three-tier Gemma 4 setup used in production is:
  - Acuifero Raspberry Pi camera node: `google/gemma-4-E4B-it` (LiteRT, int4)
  - Vigia Android handset:             `google/gemma-4-E2B-it` (LiteRT, int4)
  - Central server:                    `google/gemma-4-26B-A4B-it` (Ollama, q4_K_M)
- The Raspberry Pi 5 + LiteRT-LM proof is documented separately in the
  repository evidence, benchmark card, and demo video.
- The SINAGIR/CAP-shaped output shown here is a preview only. Nothing is
  submitted to SINAGIR production endpoints.


## 1. Dataset discovery

Path priority:

1. Kaggle attached dataset:
   `/kaggle/input/acuifero-vigia-demo-artifacts/`
2. `kagglehub.dataset_download("lucasercolano/acuifero-vigia-demo-artifacts")`
3. Local repo copy: `demo-artifacts/acuifero-vigia-demo-artifacts/`
4. Parent: `../demo-artifacts/acuifero-vigia-demo-artifacts/`

The notebook never downloads anything except via `kagglehub` (which Kaggle
permits) and only when the dataset is not already mounted.


In [ ]:
from __future__ import annotations

import csv, json, os, sys
from pathlib import Path
from pprint import pprint

try:
    from IPython.display import HTML, Image, Markdown, display
except Exception:
    class HTML:
        def __init__(self, data): self.data = data
        def __repr__(self): return str(self.data)
    class Markdown(HTML): pass
    class Image:
        def __init__(self, filename=None, width=None):
            self.filename, self.width = filename, width
        def __repr__(self): return f"Image({self.filename!r})"
    def display(x): print(x)

RUN_GEMMA = False
DATASET_SLUG = "lucasercolano/acuifero-vigia-demo-artifacts"

REQUIRED = [
    "manifest.json",
    "config/thresholds.json",
    "config/model_config.json",
    "config/prompt_templates.md",
    "outputs/logs/node_run_critical.jsonl",
    "outputs/alerts/alert_critical.json",
    "eval/expected_outputs.json",
]


def _has_all(root: Path) -> bool:
    return root.exists() and all((root / r).exists() for r in REQUIRED)


def find_dataset_root() -> Path:
    # Kaggle attached dataset.
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for child in kaggle_input.iterdir():
            if _has_all(child):
                return child
            inner = child / "acuifero-vigia-demo-artifacts"
            if _has_all(inner):
                return inner

    # Local repo copies.
    for p in [
        Path("demo-artifacts/acuifero-vigia-demo-artifacts"),
        Path("../demo-artifacts/acuifero-vigia-demo-artifacts"),
    ]:
        if _has_all(p):
            return p.resolve()

    # kagglehub fallback.
    try:
        import kagglehub
        downloaded = Path(kagglehub.dataset_download(DATASET_SLUG))
        if _has_all(downloaded):
            return downloaded
        inner = downloaded / "acuifero-vigia-demo-artifacts"
        if _has_all(inner):
            return inner
        # Some packs ship a single top-level dir; find it.
        for child in downloaded.iterdir():
            if child.is_dir() and _has_all(child):
                return child
        raise FileNotFoundError(f"kagglehub returned {downloaded} but required files missing")
    except ImportError as exc:
        raise FileNotFoundError(
            "Dataset not mounted and kagglehub not installed.\n"
            "Attach the Kaggle dataset 'lucasercolano/acuifero-vigia-demo-artifacts' "
            "or `pip install kagglehub` and re-run."
        ) from exc


DATASET_ROOT = find_dataset_root()
print(f"Dataset root: {DATASET_ROOT}")
print(f"RUN_GEMMA={RUN_GEMMA}  (default: golden fixture replay, no live model inference)")


## 2. Manifest + fixture validation

Loads `manifest.json` (the three-tier Gemma 4 demo manifest), validates every
declared path, and parses the deterministic-firewall thresholds and golden
alert outputs.


In [ ]:
def load_json(p: Path) -> dict:
    return json.loads(p.read_text(encoding="utf-8"))


def load_jsonl(p: Path) -> list[dict]:
    out = []
    for i, line in enumerate(p.read_text(encoding="utf-8").splitlines(), start=1):
        if not line.strip():
            continue
        try:
            out.append(json.loads(line))
        except json.JSONDecodeError as e:
            raise ValueError(f"Invalid JSONL at {p}:{i}: {e}") from e
    return out


manifest    = load_json(DATASET_ROOT / "manifest.json")
thresholds  = load_json(DATASET_ROOT / "config/thresholds.json")
model_cfg   = load_json(DATASET_ROOT / "config/model_config.json")
expected    = load_json(DATASET_ROOT / "eval/expected_outputs.json")

# Validate every path referenced by the manifest.
def manifest_paths(node, acc):
    if isinstance(node, dict):
        for v in node.values():
            manifest_paths(v, acc)
    elif isinstance(node, list):
        for v in node:
            manifest_paths(v, acc)
    elif isinstance(node, str) and (node.endswith((".mp4",".wav",".txt",".jsonl",".json",".png",".apk",".zip",".yaml",".md",".litert")) or node.endswith("/")):
        acc.append(node.rstrip("/"))

paths: list[str] = []
manifest_paths(manifest.get("inputs", {}),  paths)
manifest_paths(manifest.get("outputs", {}), paths)
manifest_paths(manifest.get("config", {}),  paths)
manifest_paths(manifest.get("app", {}),     paths)

missing = [p for p in paths if not (DATASET_ROOT / p).exists()]
if missing:
    raise FileNotFoundError(f"Manifest references missing paths: {missing}")

print(f"Manifest OK   submission={manifest.get('submission')}  version={manifest.get('version')}")
print(f"Model tiers   {list(model_cfg.get('tiers', {}).keys())}")
print(f"Thresholds    reference={thresholds['water_level_cm']['reference_line']}cm  "
      f"critical={thresholds['water_level_cm']['critical_line']}cm")
print(f"Expected cases: {[c['case_id'] for c in expected['cases']]}")


## 3. Show inputs

Direct inspection of replay inputs: video frames captured by the Acuifero
camera node, citizen transcripts, and Vigia screenshots.


In [ ]:
def show_table(rows, columns=None, limit=20):
    rows = rows[:limit]
    if columns is None and rows:
        columns = list(rows[0].keys())
    columns = columns or []
    header = "".join(f"<th>{c}</th>" for c in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{r.get(c,'')}</td>" for c in columns) + "</tr>"
        for r in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

# Video frames.
for case in manifest["inputs"]["video"]:
    frames_dir = case.get("frames")
    if not frames_dir:
        continue
    display(Markdown(f"**Case `{case['case']}` - frames from `{frames_dir}`**"))
    for fp in sorted((DATASET_ROOT / frames_dir).glob("*.png")):
        display(Image(filename=str(fp), width=420))

# Transcripts.
display(Markdown("**Citizen transcripts**"))
for audio in manifest["inputs"]["audio"]:
    transcript_path = DATASET_ROOT / audio["transcript"]
    text = transcript_path.read_text(encoding="utf-8").strip()
    display(Markdown(f"- **{audio['id']}**: {text}"))

# Vigia / dashboard screenshots.
display(Markdown("**Dashboard / mobile screenshots**"))
for shot in manifest["outputs"]["screenshots"]:
    display(Markdown(f"`{shot}`"))
    display(Image(filename=str(DATASET_ROOT / shot), width=520))


## 4. Deterministic replay (no LLM)

Replays the per-frame node telemetry from `outputs/logs/node_run_*.jsonl`,
re-classifies each reading against `config/thresholds.json`, and confirms the
deterministic firewall produces the expected node-state sequence.

This step is exactly what the Raspberry Pi node runs before deferring anything
to Gemma 4 - reproduced here as plain Python.


In [ ]:
def classify(reading: dict, t: dict) -> str:
    water = float(reading["water_level_cm"])
    ref   = t["water_level_cm"]["reference_line"]
    crit  = t["water_level_cm"]["critical_line"]
    if water >= crit: return "critical"
    if water >= ref:  return "vigilance"
    return "nominal"


replay = {}
for log_rel in manifest["outputs"]["logs"]:
    if not log_rel.startswith("outputs/logs/node_run_"):
        continue
    case = Path(log_rel).stem.replace("node_run_", "")
    rows = load_jsonl(DATASET_ROOT / log_rel)
    derived = [classify(r, thresholds) for r in rows]
    logged  = [r["node_state"] for r in rows]
    replay[case] = {"derived": derived, "logged": logged, "match": derived == logged}

print("Per-case node-state replay (derived vs logged):")
for case, info in replay.items():
    status = "OK" if info["match"] else "MISMATCH"
    print(f"  {case:10s}  derived={info['derived']}  logged={info['logged']}  [{status}]")

# Cross-check against eval/expected_outputs.json where a sequence is declared.
print("\nCross-check vs eval/expected_outputs.json:")
for case in expected["cases"]:
    exp_seq = case["expected"].get("node_state_sequence")
    if not exp_seq:
        continue
    cid = case["case_id"]
    short = cid.replace("river_", "").replace("evacuation_route_", "").replace("bridge_water_", "")
    if short in replay and replay[short]["derived"] == exp_seq:
        print(f"  {cid:32s} OK")
    else:
        print(f"  {cid:32s} CHECK  expected={exp_seq}  got={replay.get(short, {}).get('derived')}")


## 5. Gemma replay (golden by default)

`RUN_GEMMA = False` -> load the golden alert from `outputs/alerts/alert_critical.json`,
which captures the fused output produced by the central-node Gemma 4 26B-A4B
run on the real Raspberry Pi deployment.

`RUN_GEMMA = True` -> the notebook attempts a local LiteRT inference only when
the repo, runtime, and a model file are already present in the environment. It
**never** downloads model weights. If anything is missing it reports the error
and falls back to the golden alert.


In [ ]:
SYSTEM_PROMPT = (DATASET_ROOT / "config/prompt_templates.md").read_text(encoding="utf-8")
golden_alert  = load_json(DATASET_ROOT / "outputs/alerts/alert_critical.json")


def try_local_litert(system_prompt: str, user_payload: dict) -> dict:
    repo_backend_src = Path.cwd() / "backend" / "src"
    if repo_backend_src.exists():
        sys.path.insert(0, str(repo_backend_src))
    try:
        from acuifero_vigia.adapters.litert_node import LiteRTNodeRuntime
    except Exception as exc:
        raise RuntimeError(f"Repo LiteRT adapter not importable: {exc}") from exc
    runtime = LiteRTNodeRuntime()
    health = runtime.health()
    if not health.reachable:
        raise RuntimeError(f"Local LiteRT runtime not reachable: {health.detail}")
    payload = runtime.generate_json(system_prompt, json.dumps(user_payload, ensure_ascii=False), max_tokens=768)
    if not isinstance(payload, dict):
        raise RuntimeError("LiteRT runtime returned no parseable JSON")
    return payload


selected_alert = dict(golden_alert)
selected_alert.setdefault("runtime", {})
selected_alert["runtime"].update({
    "mode": "golden_fixture_output",
    "notebook_executes_model": False,
    "model_family": model_cfg.get("family", "Gemma 4"),
    "tiers": list(model_cfg.get("tiers", {}).keys()),
    "prompt_version": "kaggle-live-demo-v2",
})

if RUN_GEMMA:
    try:
        live = try_local_litert(SYSTEM_PROMPT, {
            "node_runs":      [load_jsonl(DATASET_ROOT / p) for p in manifest["outputs"]["logs"] if "node_run_" in p],
            "vigia_reports":  [(DATASET_ROOT / a["transcript"]).read_text(encoding="utf-8").strip() for a in manifest["inputs"]["audio"]],
            "thresholds":     thresholds,
        })
        live.setdefault("runtime", {})
        live["runtime"].update({
            "mode": "live_local_litert_inference",
            "notebook_executes_model": True,
            "model_family": model_cfg.get("family", "Gemma 4"),
            "disclaimer": "Live local inference; no internet download performed.",
        })
        selected_alert = live
        print("MODE: live_local_litert_inference")
    except Exception as exc:
        print("MODE: golden_fixture_output  (live attempt failed)")
        print(f"  {type(exc).__name__}: {exc}")
else:
    print("MODE: golden_fixture_output  (RUN_GEMMA=False)")

pprint(selected_alert["runtime"])


## 6. Final fused alert

The fused alert mirrors the production schema written to `outputs/alerts/` by
the deployed system: `final_state`, evidence list, recommended action,
`cloud_required`, `auditable`. Uncertainty above
`thresholds.deterministic_firewall.max_uncertainty_to_publish` is blocked
upstream by the deterministic firewall.


In [ ]:
level = str(selected_alert.get("final_state", "UNKNOWN")).upper()
g     = selected_alert.get("gemma_output", {})
sev   = g.get("severity", "?")
ev    = g.get("evidence", [])
act   = g.get("recommended_action", "")
unc   = g.get("uncertainty", None)

badge = {
    "NOMINAL":  "#3f8f5f",
    "WATCH":    "#c29b2e",
    "URGENT":   "#c56b2f",
    "CRITICAL": "#b4232d",
}.get(level, "#475569")

display(HTML(f'''
<div style="border:1px solid #d1d5db;border-radius:8px;padding:16px;max-width:860px">
  <div style="font-size:14px;color:#475569">Hardware replay result</div>
  <div style="font-size:28px;font-weight:700;color:{badge}">Final alert: {level}</div>
  <div style="font-size:16px;margin:6px 0 12px 0">Severity: {sev}  &middot;  Uncertainty: {unc}</div>
  <p><strong>Recommended action:</strong> {act}</p>
  <p><strong>Cloud required:</strong> {selected_alert.get("cloud_required")}  &middot;
     <strong>Auditable:</strong> {selected_alert.get("auditable")}</p>
</div>
'''))

display(Markdown("**Evidence**"))
for e in ev:
    display(Markdown(f"- {e}"))

display(Markdown("**SINAGIR/CAP-shaped preview (not submitted to production)**"))
sinagir_preview = {
    "identifier": f"acuifero-vigia-{selected_alert.get('case_id','demo')}",
    "sent":       selected_alert.get("timestamp"),
    "status":     "Exercise",
    "msgType":    "Alert",
    "scope":      "Public",
    "info": {
        "category":     "Met",
        "event":        "Flood",
        "severity":     {"nominal":"Minor","WATCH":"Moderate","URGENT":"Severe","CRITICAL":"Extreme"}.get(level, "Unknown"),
        "certainty":    "Observed" if (unc is not None and unc < 0.2) else "Likely",
        "headline":     act,
        "description":  "; ".join(ev),
    },
    "disclaimer": "Preview only. Not submitted to SINAGIR production endpoints.",
}
pprint(sinagir_preview)


## 7. Save trace

On Kaggle the trace is written to `/kaggle/working/acuifero_vigia_alert_trace.json`.
Locally it writes to the current working directory.


In [ ]:
out_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
out_path = out_dir / "acuifero_vigia_alert_trace.json"
trace = {
    "dataset_root":    str(DATASET_ROOT),
    "manifest":        {"submission": manifest.get("submission"), "version": manifest.get("version")},
    "model_tiers":     model_cfg.get("tiers", {}),
    "replay":          replay,
    "selected_alert":  selected_alert,
    "sinagir_preview": sinagir_preview,
}
out_path.write_text(json.dumps(trace, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved trace to: {out_path}")


## Acceptance notes

- Runs on Kaggle with the dataset `lucasercolano/acuifero-vigia-demo-artifacts`
  attached, or via `kagglehub.dataset_download(...)`.
- No Raspberry Pi, backend, frontend, Docker, GPU, or model download required.
- Default mode is `golden_fixture_output`. Live LiteRT is opt-in
  (`RUN_GEMMA = True`) and only runs if the runtime is already locally present.
- Three-tier Gemma 4 production setup (E4B node / E2B mobile / 26B-A4B central)
  documented in `config/model_config.json` and `config/prompt_templates.md`.
- SINAGIR/CAP output is preview only; nothing is submitted to production
  endpoints.
